<a href="https://colab.research.google.com/github/ironspiritjeff/hf-transformer-experiments/blob/main/Copy_FoodNotFood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Import necessary packages
import pprint
from pathlib import Path
import numpy as np
import torch
import datasets
from transformers import pipeline
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
try: import evaluate
except ModuleNotFoundError:
  !pip install -U evaluate
  import evaluate

# 2. Set up variables for model training and saving pipeline
DATASET_NAME = 'mrdbourke/learn_hf_food_not_food_image_captions'
MODEL_NAME = 'distilbert/distilbert-base-uncased'
MODEL_SAVE_DIR_NAME = 'models2/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# 3. Create a directory for saving models
print(f'[INFO] Creating directory for saving models: {MODEL_SAVE_DIR_NAME}')
model_save_dir = Path(MODEL_SAVE_DIR_NAME)
model_save_dir.mkdir(parents=True, exist_ok=True)

# 4. Load and preprocess dataset from Hugging Face Hub
print(f'Downloading dataset from HuggingFace Hub, name: {DATASET_NAME}')
dataset = datasets.load_dataset(DATASET_NAME)

id2label = {0: 'not_food', 1: 'food'}
label2id = {'not_food': 0, 'food': 1}

# Create a function to map IDs to labels in the dataset
def map_labels_to_number(example):
  example['label'] = label2id[example['label']]
  return example

# Map our preprocessing function to our dataset
dataset = dataset['train'].map(map_labels_to_number)

# Split the dataset into train/test sets
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

# 5. Import a tokenizer and map it to our dataset
print(f'Tokenizing text for model training with tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME,
                                          use_fast=True)

# Create a function to turn text into numbers using the above tokenizer
def tokenize_text(examples):
  return tokenizer(examples['text'],
                   padding=True,
                   truncation=True)

tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)
tokenized_dataset

# 6. Set up an evaluation metric
accuracy_metric = evaluate.load('accuracy')

def compute_accuracy(predictions_and_labels):
  predictions, labels = predictions_and_labels
  # (model will output logits in the form ([[item_1, item_2, item_3], [item_1, item_2, item_3]])), depending on the number of classes you have.
  # But we want to compare labels which are in the format ([0,0,0,0,1])
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1) # reduce the items to a single value
  return accuracy_metric.compute(predictions=predictions, references=labels)

# 7. Set up a model
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
print(f'Model loading complete!')

# Set up TrainingArguments (hyperparameters)
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.00002,           #[changed from 0.0001]
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy='epoch',
    report_to='none',
    push_to_hub=False,
    hub_private_repo=False
)

# Create Trainer Instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test'],
    processing_class=tokenizer,            # [added in]
   ### tokenizer=tokenizer, (Gemini tells me this line shouldn't be included)
    compute_metrics=compute_accuracy
)

trainer

# 8. Train the model
print(f'[INFO] Commencing model training...')
results = trainer.train()

# 9. Save the trained model to a local directory
print(f'[INFO] Model training complete, saving to local path: {model_save_dir}')
trainer.save_model(output_dir=model_save_dir)

# 10. Push the model to the Hugging Face hub
print(f'[INFO] Uploading model to Hugging Face Hub...')
model_upload_url = trainer.push_to_hub(
    commit_message='Uploading food not food text classifier model (putting it all together)'
)
print(f'[INFO] Model upload complete, model available at: {model_upload_url}')

# 11. Evaluate the model on the test data
print(f'[INFO] Performing evaluation on test dataset...')
predictions_all = trainer.predict(tokenized_dataset['test'])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f'[INFO] prediction metrics on test data:')
pprint.pprint(prediction_metrics)

[INFO] Creating directory for saving models: models2/learn_hf_food_not_food_text_classifier-distilbert-base-uncased
Tokenizing text for model training with tokenizer: distilbert/distilbert-base-uncased


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading model: distilbert/distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

  ...uncased/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...uncased/model.safetensors:   4%|4         | 12.0MB /  268MB            

[INFO] Model upload complete, model available at: https://huggingface.co/ironspiritjeff/learn_hf_food_not_food_text_classifier-distilbert-base-uncased/commit/a9f04dd98452e4ac8d8c57f5859208b8530e28b0
[INFO] Performing evaluation on test dataset...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[INFO] prediction metrics on test data:
{'test_accuracy': 1.0,
 'test_loss': 0.015194457024335861,
 'test_runtime': 2.3611,
 'test_samples_per_second': 21.177,
 'test_steps_per_second': 0.847}


In [4]:
import pandas as pd

# Get label distribution for the training set
train_labels = [example['label'] for example in dataset['train']]
train_label_counts = pd.Series(train_labels).value_counts()

# Get label distribution for the test set
test_labels = [example['label'] for example in dataset['test']]
test_label_counts = pd.Series(test_labels).value_counts()

print('Training set label distribution:')
print(train_label_counts)
print('\nTest set label distribution:')
print(test_label_counts)

Training set label distribution:
0    108
1     92
Name: count, dtype: int64

Test set label distribution:
1    33
0    17
Name: count, dtype: int64


The output above shows the distribution of labels (0 for 'not_food' and 1 for 'food') in your training and test datasets. Review these numbers to understand the class ratio and identify any significant imbalance.

In [20]:
# 12. Make sure the model works by testing it on a custom sample
from transformers import pipeline
food_not_food_classifier = pipeline(task='text-classification',
                                    model=model_save_dir,
                                    device=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'),
                                    top_k=1,
                                    batch_size=32
                                    )
food_not_food_classifier("after tacos we'll eat strawberries")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[[{'label': 'food', 'score': 0.9489285349845886}]]